# Dataproc Pipeline Testing, Monitoring & Best Practices — Demo

Companion notebook to the **Dataproc Pipeline Testing, Monitoring & Best Practices Handbook**.

This notebook covers the full loop: write a small transform module, unit-test it locally with PyTest (no cluster needed), submit the real job as a Dataproc Serverless batch configured for Persistent History Server logging, then set up a log-based failure alert.

Run the cells top to bottom. Update the variables in the **Configuration** cell first.


## 0. Setup: install & authenticate


In [ ]:
!pip install --quiet pyspark pytest google-cloud-dataproc google-cloud-storage google-cloud-logging


In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab.')
except ImportError:
    print('Not running in Colab -- make sure you have run `gcloud auth application-default login` locally.')


## 1. Configuration — edit before running


In [ ]:
PROJECT_ID = "project-2df9d2ad-b6f5-4ed5-997"
REGION = "us-central1"
BUCKET = f"{PROJECT_ID}-pipeline-demo"
EVENT_LOG_DIR = f"gs://{BUCKET}/spark-event-logs/"

print(f"Project: {PROJECT_ID} | Region: {REGION} | Bucket: {BUCKET}")


## 2. Write the transform module and unit tests


In [ ]:
transforms_code = """
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, upper

def filter_engineering(df: DataFrame) -> DataFrame:
    return df.filter(col("department") == "Engineering")

def add_department_upper(df: DataFrame) -> DataFrame:
    return df.withColumn("department_upper", upper(col("department")))

def validate_output(df: DataFrame, min_rows: int = 1) -> DataFrame:
    row_count = df.count()
    if row_count < min_rows:
        raise ValueError(f"Expected at least {min_rows} rows, got {row_count}")
    return df
"""

with open("transforms.py", "w") as f:
    f.write(transforms_code)
print("Wrote transforms.py")


In [ ]:
test_code = """
import pytest
from pyspark.sql import SparkSession
from transforms import filter_engineering, add_department_upper

@pytest.fixture(scope="session")
def spark():
    return SparkSession.builder.master("local[*]").appName("tests").getOrCreate()

def test_filter_engineering(spark):
    data = [(1, "Riya", "Engineering"), (2, "Arjun", "Sales")]
    df = spark.createDataFrame(data, ["employee_id", "full_name", "department"])
    result = filter_engineering(df)
    assert result.count() == 1
    assert result.collect()[0]["full_name"] == "Riya"

def test_add_department_upper(spark):
    data = [(1, "Engineering")]
    df = spark.createDataFrame(data, ["employee_id", "department"])
    result = add_department_upper(df)
    assert result.collect()[0]["department_upper"] == "ENGINEERING"
"""

with open("test_transforms.py", "w") as f:
    f.write(test_code)
print("Wrote test_transforms.py")


## 3. Run the unit tests locally


In [ ]:
import subprocess

result = subprocess.run(["pytest", "test_transforms.py", "-v"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    print("Tests failed -- fix before proceeding to submit the real job.")
else:
    print("All unit tests passed. No cluster or cloud cost was involved.")


## 4. Create the event-log bucket


In [ ]:
from google.cloud import storage

storage_client = storage.Client(project=PROJECT_ID)
try:
    bucket = storage_client.create_bucket(BUCKET, location=REGION)
    print(f"Bucket created: {bucket.name}")
except Exception as e:
    print(f"Bucket may already exist: {e}")


## 5. Write the job script and submit as a Dataproc Serverless batch


In [ ]:
job_code = f"""
from pyspark.sql import SparkSession
from transforms import filter_engineering, add_department_upper, validate_output

def main():
    spark = SparkSession.builder.appName("EmployeesETL").getOrCreate()
    df = spark.read.format("bigquery").option("table", "{PROJECT_ID}.hr_demo.employees").load()
    result = add_department_upper(filter_engineering(df))
    validate_output(result)
    result.write.format("bigquery") \\
        .option("table", "{PROJECT_ID}.hr_demo.testing_demo_output") \\
        .option("writeMethod", "direct") \\
        .mode("overwrite") \\
        .save()
    print("Job completed successfully.")

if __name__ == "__main__":
    main()
"""

with open("job.py", "w") as f:
    f.write(job_code)

bucket_obj = storage_client.bucket(BUCKET)
for fname in ["job.py", "transforms.py"]:
    blob = bucket_obj.blob(f"scripts/{fname}")
    blob.upload_from_filename(fname)
print("Uploaded job.py and transforms.py")


In [ ]:
from google.cloud import dataproc_v1

batch_client = dataproc_v1.BatchControllerClient(
    client_options={"api_endpoint": f"{REGION}-dataproc.googleapis.com:443"}
)

batch = dataproc_v1.Batch()
batch.pyspark_batch.main_python_file_uri = f"gs://{BUCKET}/scripts/job.py"
batch.pyspark_batch.python_file_uris = [f"gs://{BUCKET}/scripts/transforms.py"]
batch.runtime_config.version = "2.2"
# NOTE: spark.history.fs.logDirectory is a Persistent History Server (PHS) *cluster*
# setting, not a valid batch property -- setting it here causes an INVALID_ARGUMENT error.
# The batch only needs to know where to WRITE its event logs; the PHS cluster (Step 6 below)
# is separately configured to READ from that same GCS path.
batch.runtime_config.properties = {
    "spark.eventLog.enabled": "true",
    "spark.eventLog.dir": EVENT_LOG_DIR,
}

parent = f"projects/{PROJECT_ID}/locations/{REGION}"
operation = batch_client.create_batch(parent=parent, batch=batch)
print("Batch submitted, waiting for completion...")
result = operation.result()
print("Batch finished with state:", result.state)


## 5a. Stand up a Persistent History Server (PHS) cluster

This is a small, separate long-lived cluster that reads the Spark event logs the batch just wrote to `EVENT_LOG_DIR`, letting you inspect job history (DAGs, stage timings, shuffle volumes) even though the batch's own ephemeral compute is already gone. `spark.history.fs.logDirectory` belongs here, on the cluster -- not on the batch job.


In [ ]:
from google.cloud import dataproc_v1

cluster_client = dataproc_v1.ClusterControllerClient(
    client_options={"api_endpoint": f"{REGION}-dataproc.googleapis.com:443"}
)

cluster = dataproc_v1.Cluster(
    cluster_name="history-server",
    project_id=PROJECT_ID,
    config=dataproc_v1.ClusterConfig(
        endpoint_config=dataproc_v1.EndpointConfig(enable_http_port_access=True),
        software_config=dataproc_v1.SoftwareConfig(
            properties={"spark:spark.history.fs.logDirectory": EVENT_LOG_DIR}
        ),
        master_config=dataproc_v1.InstanceGroupConfig(num_instances=1),
    ),
)

try:
    op = cluster_client.create_cluster(
        request={"project_id": PROJECT_ID, "region": REGION, "cluster": cluster}
    )
    print("Creating Persistent History Server cluster (single-node)...")
    op.result()
    print("PHS cluster ready. View it: Console -> Dataproc -> Clusters -> history-server")
    print("-> Web Interfaces -> Spark History Server.")
except Exception as e:
    print(f"Cluster may already exist or creation failed: {e}")


## 6. Create a log-based metric + alerting policy for job failures

Creates the metric via the API. Finish setup by attaching an alerting policy in Console -> Monitoring -> Alerting (the metric alone doesn't notify anyone until a policy is attached).


In [ ]:
from google.cloud.logging_v2.services.metrics_service_v2 import MetricsServiceV2Client
from google.cloud.logging_v2.types import LogMetric

metrics_client = MetricsServiceV2Client()
parent = f"projects/{PROJECT_ID}"

metric = LogMetric(
    name="dataproc_job_failures",
    description="Counts failed Dataproc jobs/batches",
    filter='resource.type=("cloud_dataproc_cluster" OR "cloud_dataproc_batch") AND severity=ERROR',
)

try:
    created = metrics_client.create_log_metric(parent=parent, metric=metric)
    print(f"Log-based metric created: {created.name}")
except Exception as e:
    print(f"Metric may already exist: {e}")
print("Next: Console -> Monitoring -> Alerting -> Create Policy, using metric")
print("logging/user/dataproc_job_failures, threshold >= 1 over 5 minutes.")


## 7. Verify the output table


In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)
table = bq_client.get_table(f"{PROJECT_ID}.hr_demo.testing_demo_output")
print(f"Rows: {table.num_rows} | Created: {table.created}")


## 8. Cleanup


In [ ]:
CONFIRM_CLEANUP = True  # flip to True to actually delete resources

if CONFIRM_CLEANUP:
    bq_client.delete_table(f"{PROJECT_ID}.hr_demo.testing_demo_output", not_found_ok=True)
    blobs = list(bucket_obj.list_blobs())
    for b in blobs:
        b.delete()
    bucket_obj.delete()
    try:
        metrics_client.delete_log_metric(metric_name=f"projects/{PROJECT_ID}/metrics/dataproc_job_failures")
    except Exception as e:
        print(f"Metric deletion: {e}")
    try:
        cluster_client.delete_cluster(
            request={"project_id": PROJECT_ID, "region": REGION, "cluster_name": "history-server"}
        ).result()
        print("PHS cluster deleted.")
    except Exception as e:
        print(f"PHS cluster deletion: {e}")
    print("Output table, bucket, and log-based metric deleted.")
else:
    print("Skipped. Set CONFIRM_CLEANUP = True and re-run this cell to clean up.")
